# Process Vector Layers

This notebook enriches and prepares raw vector layers for the platform.

## Set up

In [ ]:
import subprocess
import zipfile
from pathlib import Path

import geopandas as gpd

hbl_boundary = gpd.read_file("../data/processed/vectors/HJBL_footprint.geojson")
mbtiles_dir = Path("../data/processed/vectors/mbtiles")
mbtiles_dir.mkdir(parents=True, exist_ok=True)
raw_dir = Path("../data/raw/vectors/")

### Ecological Framework

The four ecological framework layers are enriched by joining name tables from the [CANSIS Attribute Data](https://sis.agr.gc.ca/cansis/nsdb/ecostrat/index.html) site. For ecozones and ecoregions, the shapefile already contains name columns but are dropped and replaced with the full names from the attribute DBFs.

In [ ]:
shp_paths = {
    "ecozones":     raw_dir / "ecozones/Ecozones/ecozones.shp",
    "ecoregions":   raw_dir / "ecoregions/Ecoregions/ecoregions.shp",
    "ecoprovinces": raw_dir / "ecoprovinces/Ecoprovinces/ecoprovinces.shp",
    "ecodistricts": raw_dir / "ecodistricts/Ecodistricts/ecodistricts.shp",
}

dbf_paths = {
    "ecozones":     raw_dir / "zn_names.dbf",
    "ecoregions":   raw_dir / "rg_names.dbf",
    "ecoprovinces": raw_dir / "pr_names.dbf",
    "ecodistricts": raw_dir / "dt_names.dbf",
}

In [ ]:
ecozones = gpd.read_file(shp_paths["ecozones"]).drop(columns=["ZONE_NAME", "ZONE_NOM"])
zn_names = gpd.read_file(dbf_paths["ecozones"])

ecozones = ecozones.merge(zn_names, on="ECOZONE")
ecozones = gpd.clip(ecozones, hbl_boundary.to_crs(ecozones.crs))

ecozones.to_file("../data/processed/vectors/ecozones.geojson", driver="GeoJSON")

In [ ]:
ecoregions = gpd.read_file(shp_paths["ecoregions"]).drop(columns=["REGION_NAM", "REGION_NOM"])
rg_names = gpd.read_file(dbf_paths["ecoregions"])

ecoregions = ecoregions.merge(rg_names, on="ECOREGION")
ecoregions = gpd.clip(ecoregions, hbl_boundary.to_crs(ecoregions.crs))

ecoregions.to_file("../data/processed/vectors/ecoregions.geojson", driver="GeoJSON")

In [ ]:
ecoprovinces = gpd.read_file(shp_paths["ecoprovinces"])
pr_names = gpd.read_file(dbf_paths["ecoprovinces"])

pr_names["ECOPROVINC"] = pr_names["ECOPROVINC"].astype(str)
ecoprovinces = ecoprovinces.merge(pr_names, on="ECOPROVINC")
ecoprovinces = gpd.clip(ecoprovinces, hbl_boundary.to_crs(ecoprovinces.crs))

ecoprovinces.to_file("../data/processed/vectors/ecoprovinces.geojson", driver="GeoJSON")

In [ ]:
ecodistricts = gpd.read_file(shp_paths["ecodistricts"])
dt_names = gpd.read_file(dbf_paths["ecodistricts"])

ecodistricts = ecodistricts.merge(dt_names, on="ECODISTRIC")
ecodistricts = gpd.clip(ecodistricts, hbl_boundary.to_crs(ecodistricts.crs))

ecodistricts.to_file("../data/processed/vectors/ecodistricts.geojson", driver="GeoJSON")

### Mining Claims

Extract the `mlas_operational_gis_data.zip` archive (not extracted during download) and clip the four mining layers to the HBL boundary.

In [ ]:
zip_path = raw_dir / "mlas_operational_gis_data.zip"
extract_dir = raw_dir / "mlas_operational_gis_data"

if not extract_dir.exists():
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(extract_dir)
    print(f"Extracted → {extract_dir}")
else:
    print(f"Already extracted: {extract_dir}")

In [ ]:
mining_dir = extract_dir / "MLAS_Operational_GIS_Data"
output_dir = Path("../data/processed/vectors/")

mining_layers = [
    "Mining_Land_Tenure",
    "Non_Mining_Land_Tenure",
    "Operational_Cell_Claims",
    "Operational_Alienations",
    "Plans_Permits",
]

for layer_name in mining_layers:
    gdf = gpd.read_file(mining_dir / f"{layer_name}.shp")
    clipped = gpd.clip(gdf, hbl_boundary.to_crs(gdf.crs))
    output_path = output_dir / f"{layer_name}.geojson"
    clipped.to_file(output_path, driver="GeoJSON")
    print(f"{layer_name}: {len(gdf)} → {len(clipped)} features")

### Generate MBTiles

Create MBTiles from the clipped GeoJSONs using tippecanoe. All features are retained at every zoom level (no dropping) starting from zoom 3.

In [ ]:
geojson_dir = Path("../data/processed/vectors/")

geojson_files = sorted(geojson_dir.glob("*.geojson"))
# Exclude the HBL footprint files — they are not tileset layers
geojson_files = [f for f in geojson_files if "HBL_footprint" not in f.name]

for geojson_path in geojson_files:
    layer_name = geojson_path.stem
    output_path = mbtiles_dir / f"{layer_name}.mbtiles"

    cmd = [
        "tippecanoe",
        "-o", str(output_path),
        "-Z", "3",
        "-z", "14",
        "-r1",
        "--no-feature-limit",
        "--no-tile-size-limit",
        f"--layer={layer_name}",
        "--force",
        str(geojson_path),
    ]
    print(f"{layer_name}: generating MBTiles...")
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"  ERROR: {result.stderr}")
    else:
        print(f"  → {output_path}")